
# WaveDetect Split-Robustness Uncertainty Summary

The 20 WaveDetect train/test splits overlap, so they are **not independent replications**.
Accordingly, this notebook does not report a conventional confidence interval across split means.

Instead it reports, for each metric/model comparison:

- mean delta
- median delta
- interquartile range
- minimum and maximum delta
- number of positive splits out of 20

This provides uncertainty/dispersion information without pretending that the 20 overlapping splits are iid samples.

The paper's main comparison is `W+C+F` versus `W`.


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
candidates = [HERE, HERE.parent, HERE.parent.parent]

REPRO_ROOT = None
for p in candidates:
    if (p / "results" / "wavedetect_exact20_deltas.csv").exists():
        REPRO_ROOT = p
        break
    if (p / "reproduction" / "results" / "wavedetect_exact20_deltas.csv").exists():
        REPRO_ROOT = p / "reproduction"
        break

if REPRO_ROOT is None:
    raise FileNotFoundError(
        "Could not locate reproduction/results/wavedetect_exact20_deltas.csv"
    )

DELTA_PATH = REPRO_ROOT / "results" / "wavedetect_exact20_deltas.csv"
OUT_PATH = REPRO_ROOT / "results" / "wavedetect_exact20_dispersion_summary.csv"

print("REPRO_ROOT:", REPRO_ROOT)
print("DELTA_PATH:", DELTA_PATH)


REPRO_ROOT: ..
DELTA_PATH: ../results/wavedetect_exact20_deltas.csv


In [2]:

d = pd.read_csv(DELTA_PATH)

print("shape:", d.shape)
print("columns:", d.columns.tolist())
display(d.head())


shape: (100, 7)
columns: ['seed', 'method', 'delta_accuracy', 'delta_f1', 'delta_auroc', 'delta_auprc', 'delta_tpr_at_1fpr']


,seed,method,delta_accuracy,delta_f1,delta_auroc,delta_auprc,delta_tpr_at_1fpr
0,0,W+C,0.007854,0.007048,0.000332,0.000752,0.044567
1,0,W+F,0.016337,0.014163,0.004848,0.000482,0.030203
2,0,W+C+F,0.022306,0.018947,0.004886,0.000991,0.104972
3,0,W+R,0.020735,0.017879,0.005105,0.000187,0.012155
4,0,W+C+R,0.030474,0.026150,0.005200,0.000833,0.071455



## Normalize schema

The cell below handles the common output formats from the exact-20 notebook.


In [3]:
spec_col = "method"
seed_col = "seed"

metric_cols = {
    "AUROC": "delta_auroc",
    "TPR@1%FPR": "delta_tpr_at_1fpr",
    "F1": "delta_f1",
    "AUPRC": "delta_auprc",
}

print("spec column:", spec_col)
print("metric columns:", metric_cols)

spec column: method
metric columns: {'AUROC': 'delta_auroc', 'TPR@1%FPR': 'delta_tpr_at_1fpr', 'F1': 'delta_f1', 'AUPRC': 'delta_auprc'}


In [4]:

rows = []

for spec, sub in d.groupby(spec_col):
    for metric, col in metric_cols.items():
        x = pd.to_numeric(sub[col], errors="coerce").dropna().to_numpy()

        if len(x) == 0:
            continue

        q25, q75 = np.quantile(x, [0.25, 0.75])

        rows.append({
            "spec": spec,
            "metric": metric,
            "n_splits": len(x),
            "mean": x.mean(),
            "median": np.median(x),
            "q25": q25,
            "q75": q75,
            "iqr": q75 - q25,
            "min": x.min(),
            "max": x.max(),
            "n_positive": int((x > 0).sum()),
            "positive_fraction": float((x > 0).mean()),
        })

summary = pd.DataFrame(rows)

display(summary)
summary.to_csv(OUT_PATH, index=False)
print("saved:", OUT_PATH)


,spec,metric,n_splits,mean,median,q25,q75,iqr,min,max,n_positive,positive_fraction
0,W+C,AUROC,20,0.000975,0.000903,0.000533,0.001492,0.000959,-0.000368,0.002192,18,0.90
1,W+C,TPR@1%FPR,20,0.020737,0.019521,0.007827,0.045672,0.037845,-0.068140,0.061878,16,0.80
2,W+C,F1,20,0.005896,0.005543,0.005243,0.006715,0.001472,0.002234,0.008733,20,1.00
3,W+C,AUPRC,20,0.000793,0.000764,0.000522,0.001046,0.000524,0.000231,0.001478,20,1.00
4,W+C+F,AUROC,20,0.004984,0.004900,0.004701,0.005544,0.000843,0.003033,0.006596,20,1.00
5,W+C+F,TPR@1%FPR,20,0.071786,0.072744,0.047790,0.097790,0.050000,0.013260,0.126335,20,1.00
6,W+C+F,F1,20,0.019996,0.020080,0.018471,0.021784,0.003313,0.016663,0.023140,20,1.00
7,W+C+F,AUPRC,20,0.001925,0.001865,0.001542,0.002383,0.000841,0.000297,0.003179,20,1.00
8,W+C+R,AUROC,20,0.005588,0.005579,0.004793,0.006284,0.001491,0.003696,0.007679,20,1.00
9,W+C+R,TPR@1%FPR,20,0.073444,0.068140,0.057182,0.091344,0.034162,0.041621,0.125967,20,1.00


saved: ../results/wavedetect_exact20_dispersion_summary.csv



## Main-paper `W+C+F` summary

This cell searches for the `W+C+F` feature set and prints paper-ready dispersion statistics.


In [5]:

spec_values = d[spec_col].astype(str).unique().tolist()
print("available specs:", spec_values)

def canonical(s):
    return (
        str(s)
        .lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace("+", "")
    )

target = None
for s in spec_values:
    cs = canonical(s)
    if "w" in cs and "c" in cs and "f" in cs and "r" not in cs:
        target = s
        break

if target is None:
    print("Could not identify W+C+F automatically.")
    print("Set target manually to one of the available specs above.")
else:
    print("Using target spec:", target)
    main = summary[summary["spec"] == target].copy()
    display(main)

    print("\nPAPER-READY DISPERSION SUMMARY")
    print("--------------------------------")

    for _, r in main.iterrows():
        print(
            f"{r['metric']:10s} "
            f"mean={r['mean']:+.6f}, "
            f"median={r['median']:+.6f}, "
            f"IQR=[{r['q25']:+.6f}, {r['q75']:+.6f}], "
            f"range=[{r['min']:+.6f}, {r['max']:+.6f}], "
            f"positive={int(r['n_positive'])}/{int(r['n_splits'])}"
        )


available specs: ['W+C', 'W+F', 'W+C+F', 'W+R', 'W+C+R']
Using target spec: W+C+F


,spec,metric,n_splits,mean,median,q25,q75,iqr,min,max,n_positive,positive_fraction
4,W+C+F,AUROC,20,0.004984,0.004900,0.004701,0.005544,0.000843,0.003033,0.006596,20,1.0
5,W+C+F,TPR@1%FPR,20,0.071786,0.072744,0.047790,0.097790,0.050000,0.013260,0.126335,20,1.0
6,W+C+F,F1,20,0.019996,0.020080,0.018471,0.021784,0.003313,0.016663,0.023140,20,1.0
7,W+C+F,AUPRC,20,0.001925,0.001865,0.001542,0.002383,0.000841,0.000297,0.003179,20,1.0



PAPER-READY DISPERSION SUMMARY
--------------------------------
AUROC      mean=+0.004984, median=+0.004900, IQR=[+0.004701, +0.005544], range=[+0.003033, +0.006596], positive=20/20
TPR@1%FPR  mean=+0.071786, median=+0.072744, IQR=[+0.047790, +0.097790], range=[+0.013260, +0.126335], positive=20/20
F1         mean=+0.019996, median=+0.020080, IQR=[+0.018471, +0.021784], range=[+0.016663, +0.023140], positive=20/20
AUPRC      mean=+0.001925, median=+0.001865, IQR=[+0.001542, +0.002383], range=[+0.000297, +0.003179], positive=20/20



## Interpretation

Because the same examples can appear in multiple held-out splits, the 20 split deltas are correlated.
The appropriate claim is therefore **split robustness**, not a 95% confidence interval over 20 independent trials.

If all 20 deltas remain positive and the IQR is comfortably above zero, the paper can report:

> Across 20 overlapping protocol-matched splits, the gain is positive on every split; the median and interquartile range show that the improvement is not driven by a small number of favorable partitions.

If per-example prediction files are available from the exact-20 experiment, a separate paired bootstrap over evaluation examples can be added later for a true sampling-based interval.
